# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [ ]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

## 2. Pipeline de procesamiento

In [ ]:
feature_columns = contract["features"]

# TODO: crea un preprocesador para árboles con imputación por mediana
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns),
    ]
)

# TODO: crea otro con imputación y StandardScaler
scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            feature_columns,
        ),
    ]
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Escribe aquí tu respuesta.

**Respuesta:**

Si el `StandardScaler` se ajustara con todo `X_train` antes de la validación cruzada, la media y la desviación estándar se calcularían usando también las filas que luego quedan en el fold de validación. Así, información del fold de validación **se filtraría** al entrenamiento (data leakage) y la métrica de CV saldría optimista, porque el modelo "vio" indirectamente los datos con los que se evalúa.

Dentro del `Pipeline`, en cada fold el scaler se ajusta (`fit`) únicamente con las filas de entrenamiento de ese fold y solo se aplica (`transform`) al fold de validación, igual que ocurrirá en producción con datos nuevos. Además, el pipeline guarda el scaler junto al modelo, lo que garantiza que en inferencia se aplique exactamente la misma transformación.

## 3. Modelos candidatos

In [ ]:
# TODO: crea pipelines para DecisionTree, LogisticRegression y KNN
models = {
    "decision_tree": Pipeline(
        steps=[
            ("preprocess", clone(tree_preprocessor)),
            ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]
    ),
    "logistic_regression": Pipeline(
        steps=[
            ("preprocess", clone(scaled_preprocessor)),
            ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
        ]
    ),
    "knn": Pipeline(
        steps=[
            ("preprocess", clone(scaled_preprocessor)),
            ("model", KNeighborsClassifier()),
        ]
    ),
}

list(models)

## 4. Comparación inicial con validación cruzada

In [ ]:
# TODO: usa StratifiedKFold de 5 folds y cross_validate
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    rows.append(
        {
            "model": name,
            # Reporta accuracy promedio de train, accuracy promedio de validation y desviación estándar
            "train_accuracy": scores["train_score"].mean(),
            "val_accuracy": scores["test_score"].mean(),
            "val_std": scores["test_score"].std(),
        }
    )

# Guarda el resumen en initial_comparison
initial_comparison = (
    pd.DataFrame(rows)
    .set_index("model")
    .sort_values("val_accuracy", ascending=False)
)
initial_comparison.round(4)

## 5. Optimización de hiperparámetros

In [ ]:
# TODO: define espacios pequeños de búsqueda para los tres modelos
param_grids = {
    "decision_tree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 3, 5],
    },
    "logistic_regression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
    },
    "knn": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
    },
}

# TODO: ejecuta GridSearchCV con scoring="accuracy"
searches = {}
rows = []

for name, pipeline in models.items():
    search = GridSearchCV(
        estimator=clone(pipeline),
        param_grid=param_grids[name],
        scoring="accuracy",
        cv=cv,
        return_train_score=True,
        refit=True,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    searches[name] = search

    best = search.best_index_
    rows.append(
        {
            "model": name,
            "train_accuracy": search.cv_results_["mean_train_score"][best],
            "val_accuracy": search.best_score_,
            "val_std": search.cv_results_["std_test_score"][best],
            "best_params": search.best_params_,
        }
    )

# Guarda las búsquedas en searches y el resumen en tuned_comparison
tuned_comparison = (
    pd.DataFrame(rows)
    .set_index("model")
    .sort_values("val_accuracy", ascending=False)
)
tuned_comparison

## 6. Selección y persistencia del campeón

In [ ]:
# TODO: selecciona el mayor best_score_ de CV
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_  # pipeline completo, ya reajustado con todo train

print("Campeón:", champion_name)
print("Accuracy CV:", round(searches[champion_name].best_score_, 4))
print("Mejores hiperparámetros:", searches[champion_name].best_params_)

# TODO: guarda champion_model.joblib, training_metadata.json y cv_results.csv
import sklearn

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

champion_search = searches[champion_name]
best_idx = champion_search.best_index_

training_metadata = {
    "champion": champion_name,
    "cv_accuracy": float(champion_search.best_score_),
    "cv_std": float(champion_search.cv_results_["std_test_score"][best_idx]),
    "train_accuracy_cv": float(champion_search.cv_results_["mean_train_score"][best_idx]),
    "best_params": champion_search.best_params_,
    "cv": {"type": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": RANDOM_STATE},
    "scoring": "accuracy",
    "random_state": RANDOM_STATE,
    "features": feature_columns,
    "target": contract["target"],
    "n_train_samples": int(len(X_train)),
    "sklearn_version": sklearn.__version__,
}
with open(ARTIFACTS_DIR / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, indent=2, ensure_ascii=False, default=str)

cv_results = pd.concat(
    [
        initial_comparison.assign(stage="initial", best_params=None),
        tuned_comparison.assign(stage="tuned"),
    ]
).reset_index()
cv_results["best_params"] = cv_results["best_params"].astype(str)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

cv_results.round(4)

In [ ]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.

**Respuesta** (resultados con el dataset de vinos estándar; si tu CSV difiere, las cifras cambiarán un poco):

Elegí la **regresión logística** (`C=1`, con imputación y `StandardScaler` dentro del pipeline).

| Modelo | Accuracy train (CV) | Accuracy validación (CV) | Desv. estándar |
|---|---:|---:|---:|
| Regresión logística | 1.000 | **0.979** | 0.028 |
| KNN (k=7, uniform) | 0.965 | 0.958 | 0.041 |
| Árbol de decisión (depth=4) | 0.998 | 0.902 | 0.059 |

- **Desempeño:** tiene el mayor accuracy medio de validación cruzada (≈ 0.979), por encima de KNN (≈ 0.958) y del árbol (≈ 0.902), y la selección se hizo solo con `best_score_` de CV, sin usar test.
- **Variabilidad:** también tiene la menor desviación entre folds (≈ 0.028 frente a 0.041 y 0.059), así que su desempeño es más estable.
- **Sobreajuste:** el árbol tiene accuracy de train ≈ 1.0 frente a ≈ 0.90 en validación (brecha ≈ 0.10), una señal clara de sobreajuste. En la regresión logística también el train es 1.0, con una brecha pequeña (≈ 0.02) y regularización; el KNN tiene la brecha más reducida, pero rinde menos.
- **Complejidad e interpretabilidad:** es un modelo lineal simple y rápido, con coeficientes interpretables por variable (con la salvedad de la correlación entre variables fenólicas). El árbol es interpretable pero inestable; el KNN guarda todo el conjunto de entrenamiento y es más costoso en inferencia.
- **Limitación:** con solo 142 muestras de entrenamiento, las diferencias de ≈ 0.02 entre logística y KNN están dentro de una desviación estándar de la CV, por lo que la ventaja sobre KNN no es concluyente. Se elige la logística por rendimiento, estabilidad y simplicidad.

## Checklist

- [ ] Todo el procesamiento está dentro de pipelines.
- [ ] Comparé tres familias de modelos.
- [ ] Optimicé hiperparámetros con CV.
- [ ] Elegí el campeón sin usar test.
- [ ] Guardé el pipeline completo y sus metadatos.